# 05 — Polarimetric Decomposition & Analysis

Quad-pol polarimetric analysis of NISAR GCOV data: build coherency matrices,
run Freeman-Durden and Cloude-Pottier decompositions, compute polarimetric
indices, and validate scientific correctness.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bullocke/nice-sar/blob/main/notebooks/05_polarimetric_analysis.ipynb)

## 1. Install and Import

In [ ]:
%pip install -q "fsspec<=2025.3.0" "s3fs<=2025.3.0" git+https://github.com/bullocke/nice-sar.git

In [ ]:
import logging
import os

import matplotlib
if not os.environ.get("DISPLAY"):
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from nice_sar.auth.earthdata import login, get_s3_filesystem, get_https_filesystem
from nice_sar.io.hdf5 import open_nisar, get_frequencies, get_polarizations
from nice_sar.io.products import read_gcov, read_quad_covariances, get_projection_info
from nice_sar.preprocess.calibration import linear_to_db
from nice_sar.analysis.decomposition import (
    build_coherency_matrix,
    freeman_durden,
    cloude_pottier,
    compute_pauli_rgb,
)
from nice_sar.analysis.polarimetry import (
    compute_span,
    compute_rfdi,
    volume_proxy,
    compute_indices,
)
from nice_sar.viz.display import percentile_stretch

logging.basicConfig(level=logging.INFO, format="%(name)s — %(message)s", force=True)
logger = logging.getLogger(__name__)

# ── Study Areas ──────────────────────────────────────────────────────────────
STUDY_AREAS = {
    "salt_lake_city": (-112.1, 40.5, -111.7, 40.9),
    "cascades_wa": (-122.5, 46.0, -121.5, 47.0),
    "amazon": (-55.0, -3.5, -54.0, -2.5),
}
STUDY_AREA = "salt_lake_city"  # ← change to try other regions
bbox = STUDY_AREAS[STUDY_AREA]
logger.info("Study area: %s  bbox=%s", STUDY_AREA, bbox)

## 2. Authenticate and Load Quad-Pol Data

This notebook requires **quad-pol** GCOV data (HH, HV, VH, VV) for full
polarimetric decomposition. If only dual-pol is available, the decomposition
sections will be skipped and we'll fall back to dual-pol indices.

In [ ]:
try:
    login()
except Exception as e:
    logger.error(
        "Authentication failed: %s\n"
        "Ensure credentials are configured via one of:\n"
        "  1. Environment variables: EARTHDATA_USERNAME / EARTHDATA_PASSWORD\n"
        "  2. ~/.netrc with: machine urs.earthdata.nasa.gov login <user> password <pass>\n"
        "  3. Interactive prompt (Colab / Jupyter only)",
        e,
    )
    raise

if os.environ.get("AWS_DEFAULT_REGION") == "us-west-2":
    fs = get_s3_filesystem()
else:
    fs = get_https_filesystem()

# Search for a GCOV granule
import earthaccess

results = earthaccess.search_data(
    short_name="NISAR_L2_GCOV_BETA_V1",
    bounding_box=bbox,
    count=5,
)
links = earthaccess.results.DataGranule.data_links(results[0], access="direct")
if not links:
    links = earthaccess.results.DataGranule.data_links(results[0], access="external")
granule_url = links[0]
logger.info("Using: %s", granule_url.split("/")[-1])

In [ ]:
# Open file and check polarization availability
h5 = open_nisar(granule_url, filesystem=fs)
pols = get_polarizations(h5, "A")
logger.info("Available polarizations: %s", pols)

is_quad_pol = all(p in pols for p in ("HHHH", "HVHV", "VVVV"))
logger.info("Quad-pol available: %s", is_quad_pol)

# Read HH and HV (always available)
da_hh = read_gcov(h5, frequency="A", polarization="HH")
da_hv = read_gcov(h5, frequency="A", polarization="HV")

hh_full = da_hh.values
hv_full = da_hv.values

# Try VV if available
try:
    da_vv = read_gcov(h5, frequency="A", polarization="VV")
    vv_full = da_vv.values
    logger.info("VV loaded — full quad-pol analysis enabled")
except Exception:
    vv_full = None
    logger.warning("VV not available — falling back to dual-pol analysis")

# Take a center 1024×1024 subset for processing speed
cy, cx = hh_full.shape[0] // 2, hh_full.shape[1] // 2
sz = 512
slc = (slice(cy - sz, cy + sz), slice(cx - sz, cx + sz))
hh = hh_full[slc]
hv = hv_full[slc]
vv = vv_full[slc] if vv_full is not None else None

logger.info("Subset shape: %s", hh.shape)

## 3. Polarimetric Indices (Dual-Pol)

These indices work with any dual-pol (HH/HV) data:

- **SPAN**: Total backscatter power
- **RFDI**: Radar Forest Degradation Index — positive = surface, negative = volume
- **Volume Proxy**: HV / (HH + HV) — higher = more volume scattering
- **HH/HV Ratio**: Co-pol to cross-pol ratio (indicator of scattering mechanism)

In [ ]:
# Compute polarimetric indices
span = compute_span(hh=hh, hv=hv, vv=vv)
rfdi = compute_rfdi(hh=hh, hv=hv, vv=vv)
vol = volume_proxy(hh, hv)
indices = compute_indices(hh, hv=hv, vv=vv)

logger.info("SPAN range: %.4f – %.4f", np.nanmin(span), np.nanmax(span))
logger.info("RFDI range: %.4f – %.4f", np.nanmin(rfdi), np.nanmax(rfdi))
logger.info("Volume proxy range: %.4f – %.4f", np.nanmin(vol), np.nanmax(vol))
logger.info("Indices computed: %s", list(indices.keys()))

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

im0 = axes[0, 0].imshow(linear_to_db(span), cmap="viridis", aspect="equal")
axes[0, 0].set_title("SPAN (dB)")
plt.colorbar(im0, ax=axes[0, 0], shrink=0.8)

im1 = axes[0, 1].imshow(rfdi, cmap="RdYlGn", vmin=-1, vmax=1, aspect="equal")
axes[0, 1].set_title("RFDI (Forest Degradation Index)")
plt.colorbar(im1, ax=axes[0, 1], shrink=0.8)

im2 = axes[1, 0].imshow(vol, cmap="YlGn", vmin=0, vmax=0.5, aspect="equal")
axes[1, 0].set_title("Volume Proxy (HV / (HH+HV))")
plt.colorbar(im2, ax=axes[1, 0], shrink=0.8)

if "hh_hv_ratio_db" in indices:
    im3 = axes[1, 1].imshow(indices["hh_hv_ratio_db"], cmap="plasma", aspect="equal")
    axes[1, 1].set_title("HH/HV Ratio (dB)")
    plt.colorbar(im3, ax=axes[1, 1], shrink=0.8)
else:
    axes[1, 1].set_visible(False)

fig.suptitle(f"Polarimetric Indices — {STUDY_AREA}", fontsize=14)
plt.tight_layout()
plt.savefig("polarimetric_indices.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Quad-Pol Covariance Matrix

Read the full covariance matrix elements (HHHH, HVHV, VVVV, HHHV, HHVV, HVVV)
and build the 3×3 coherency matrix **T** via the Pauli basis transformation:

$$\mathbf{T} = \mathbf{U} \cdot \mathbf{C} \cdot \mathbf{U}^H$$

where **U** is the Pauli unitary matrix and **C** is the lexicographic covariance matrix.

> ⚠️ This section requires **quad-pol** data. If only dual-pol is available, it will be skipped.

In [ ]:
if not is_quad_pol:
    logger.warning("Quad-pol not available — skipping coherency matrix and decompositions.")
    T = None
else:
    # Read covariance elements
    grid_path = "/science/LSAR/GCOV/grids/frequencyA"
    covariances_full = read_quad_covariances(h5, grid_path)
    logger.info("Covariance terms loaded: %s", list(covariances_full.keys()))

    # Subset covariance elements to match our spatial window
    covariances = {k: v[slc] for k, v in covariances_full.items()}

    # Build coherency matrix T (3×3×H×W) with spatial averaging
    T = build_coherency_matrix(covariances, window=7)
    logger.info("Coherency matrix shape: %s", T.shape)

    # Verify Hermitian symmetry
    T_h = np.conj(np.transpose(T, (1, 0, 2, 3)))
    hermitian_error = np.max(np.abs(T - T_h))
    logger.info("Hermitian symmetry error: %.2e (should be ~0)", hermitian_error)

## 5. Freeman-Durden 3-Component Decomposition

Decomposes the coherency matrix into three scattering mechanisms:

| Component | Symbol | Physical Meaning |
|-----------|--------|------------------|
| Surface   | Ps     | Single-bounce (bare soil, water) |
| Double-bounce | Pd | Dihedral (buildings, tree trunks + ground) |
| Volume    | Pv     | Random (forest canopy, vegetation) |

**Scientific validation**: Ps + Pd + Pv should approximate the total SPAN.

In [ ]:
if T is not None:
    Ps, Pd, Pv = freeman_durden(T)

    logger.info("Surface (Ps):  min=%.4f  max=%.4f  mean=%.4f", np.nanmin(Ps), np.nanmax(Ps), np.nanmean(Ps))
    logger.info("Double (Pd):   min=%.4f  max=%.4f  mean=%.4f", np.nanmin(Pd), np.nanmax(Pd), np.nanmean(Pd))
    logger.info("Volume (Pv):   min=%.4f  max=%.4f  mean=%.4f", np.nanmin(Pv), np.nanmax(Pv), np.nanmean(Pv))

    # ── Scientific Validation: Power Conservation ──
    total_decomp = Ps + Pd + Pv
    span_from_T = T[0, 0].real + T[1, 1].real + T[2, 2].real

    ratio = total_decomp / (span_from_T + 1e-12)
    logger.info("Power conservation — Median(Ps+Pd+Pv / SPAN) = %.4f", np.nanmedian(ratio))
    logger.info("  (Should be reasonably close to 1.0 for the simplified model)")

    # RGB: R=Pd (double-bounce), G=Pv (volume), B=Ps (surface)
    fd_rgb = np.stack([
        percentile_stretch(Pd),
        percentile_stretch(Pv),
        percentile_stretch(Ps),
    ], axis=-1)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(percentile_stretch(Ps), cmap="Blues", aspect="equal")
    axes[0].set_title("Ps (Surface)")

    axes[1].imshow(percentile_stretch(Pd), cmap="Reds", aspect="equal")
    axes[1].set_title("Pd (Double-bounce)")

    axes[2].imshow(percentile_stretch(Pv), cmap="Greens", aspect="equal")
    axes[2].set_title("Pv (Volume)")

    axes[3].imshow(fd_rgb, aspect="equal")
    axes[3].set_title("Freeman-Durden RGB\nR=Pd  G=Pv  B=Ps")

    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])

    fig.suptitle(f"Freeman-Durden Decomposition — {STUDY_AREA}", fontsize=14)
    plt.tight_layout()
    plt.savefig("freeman_durden.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    logger.info("Skipping Freeman-Durden (requires quad-pol)")

## 6. Cloude-Pottier H/A/α Decomposition

Eigenvalue decomposition of the coherency matrix yields:

- **Entropy (H)**: Randomness of scattering — 0 = single mechanism, 1 = random
- **Anisotropy (A)**: Relative importance of 2nd vs 3rd eigenvalue
- **Alpha (α)**: Mean scattering angle — 0° = surface, 45° = dipole, 90° = dihedral

**Physical constraints**: H ∈ [0, 1], A ∈ [0, 1], α ∈ [0°, 90°]

In [ ]:
if T is not None:
    H, A, alpha = cloude_pottier(T)

    valid = np.isfinite(H)
    logger.info("Entropy (H):    min=%.4f  max=%.4f  mean=%.4f", np.nanmin(H), np.nanmax(H), np.nanmean(H))
    logger.info("Anisotropy (A): min=%.4f  max=%.4f  mean=%.4f", np.nanmin(A), np.nanmax(A), np.nanmean(A))
    logger.info("Alpha (°):      min=%.2f  max=%.2f  mean=%.2f", np.nanmin(alpha), np.nanmax(alpha), np.nanmean(alpha))

    # ── Scientific Validation: Physical Range Checks ──
    assert np.all(H[valid] >= 0) and np.all(H[valid] <= 1), "H out of [0, 1]!"
    assert np.all(A[valid] >= 0) and np.all(A[valid] <= 1), "A out of [0, 1]!"
    assert np.all(alpha[valid] >= 0) and np.all(alpha[valid] <= 90), "alpha out of [0, 90]!"
    logger.info("✓ All physical range checks passed")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    im0 = axes[0].imshow(H, cmap="inferno", vmin=0, vmax=1, aspect="equal")
    axes[0].set_title("Entropy (H)")
    plt.colorbar(im0, ax=axes[0], shrink=0.8)

    im1 = axes[1].imshow(A, cmap="magma", vmin=0, vmax=1, aspect="equal")
    axes[1].set_title("Anisotropy (A)")
    plt.colorbar(im1, ax=axes[1], shrink=0.8)

    im2 = axes[2].imshow(alpha, cmap="twilight", vmin=0, vmax=90, aspect="equal")
    axes[2].set_title("Alpha (°)")
    plt.colorbar(im2, ax=axes[2], shrink=0.8)

    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])

    fig.suptitle(f"Cloude-Pottier H/A/α — {STUDY_AREA}", fontsize=14)
    plt.tight_layout()
    plt.savefig("cloude_pottier.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    logger.info("Skipping Cloude-Pottier (requires quad-pol)")

## 7. H-α Plane Classification

The Cloude-Pottier H-α plane partitions scattering into 9 zones based on
entropy and alpha angle. This is the standard classification scheme for
polarimetric SAR interpretation.

| Zone | H range | α range | Interpretation |
|------|---------|---------|----------------|
| Z1 | Low | Low | Surface (Bragg) |
| Z2 | Low | Medium | Dipole |
| Z5 | Medium | Low | Medium entropy surface |
| Z9 | High | High | High entropy dihedral |

In [ ]:
if T is not None:
    fig, ax = plt.subplots(figsize=(10, 8))

    # Scatter H vs alpha (subsample for speed)
    h_flat = H[valid].ravel()
    a_flat = alpha[valid].ravel()
    step = max(1, len(h_flat) // 50000)  # subsample to ~50k points
    ax.scatter(
        h_flat[::step], a_flat[::step],
        s=0.5, alpha=0.3, c="steelblue", edgecolors="none",
    )

    # Draw Cloude-Pottier zone boundaries
    ax.axhline(y=42.5, color="gray", linestyle="--", linewidth=0.8)
    ax.axhline(y=47.5, color="gray", linestyle="--", linewidth=0.8)
    ax.axvline(x=0.5, color="gray", linestyle="--", linewidth=0.8)
    ax.axvline(x=0.9, color="gray", linestyle="--", linewidth=0.8)

    # Zone labels
    zone_labels = [
        (0.25, 21, "Z1\nSurface"), (0.25, 45, "Z2\nDipole"), (0.25, 68, "Z3\nDihedral"),
        (0.7, 21, "Z4"), (0.7, 45, "Z5"), (0.7, 68, "Z6"),
        (0.95, 21, "Z7"), (0.95, 45, "Z8"), (0.95, 68, "Z9\nRandom"),
    ]
    for x, y, label in zone_labels:
        ax.text(x, y, label, ha="center", va="center", fontsize=8,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.7))

    ax.set_xlabel("Entropy (H)", fontsize=12)
    ax.set_ylabel("Alpha (°)", fontsize=12)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 90)
    ax.set_title(f"H-α Plane — {STUDY_AREA}", fontsize=14)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("h_alpha_plane.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    logger.info("Skipping H-alpha plane (requires quad-pol)")

## 8. Pauli RGB Composite (Quad-Pol)

The Pauli decomposition maps three scattering mechanisms to RGB:

- **Red** = |HH − VV| (double-bounce)
- **Green** = √2 · |HV| (volume)
- **Blue** = |HH + VV| (surface)

In [ ]:
if vv is not None:
    pauli = compute_pauli_rgb(hh, hv, vv)
    logger.info("Pauli RGB shape: %s", pauli.shape)

    # Stretch each band for display
    pauli_display = np.stack([
        percentile_stretch(pauli[0]),
        percentile_stretch(pauli[1]),
        percentile_stretch(pauli[2]),
    ], axis=-1)

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(pauli_display, aspect="equal")
    ax.set_title(f"Pauli RGB — R=|HH-VV|  G=√2·|HV|  B=|HH+VV|\n{STUDY_AREA}", fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()
    plt.savefig("pauli_rgb.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    logger.info("Skipping Pauli RGB (requires VV)")

## 9. Comparative Summary

Side-by-side comparison of all decomposition and index products to show how
different polarimetric representations highlight different features in the landscape.

In [ ]:
n_cols = 3 if T is not None else 2
fig, axes = plt.subplots(2, n_cols, figsize=(6 * n_cols, 10))

# Row 1: Indices
axes[0, 0].imshow(linear_to_db(span), cmap="viridis", aspect="equal")
axes[0, 0].set_title("SPAN (dB)")
axes[0, 1].imshow(rfdi, cmap="RdYlGn", vmin=-1, vmax=1, aspect="equal")
axes[0, 1].set_title("RFDI")

if T is not None:
    axes[0, 2].imshow(fd_rgb, aspect="equal")
    axes[0, 2].set_title("Freeman-Durden RGB")

# Row 2: More products
axes[1, 0].imshow(vol, cmap="YlGn", vmin=0, vmax=0.5, aspect="equal")
axes[1, 0].set_title("Volume Proxy")

if T is not None:
    axes[1, 1].imshow(H, cmap="inferno", vmin=0, vmax=1, aspect="equal")
    axes[1, 1].set_title("Entropy (H)")
    axes[1, 2].imshow(pauli_display if vv is not None else H, aspect="equal")
    axes[1, 2].set_title("Pauli RGB" if vv is not None else "Alpha (°)")
else:
    if "hh_hv_ratio_db" in indices:
        axes[1, 1].imshow(indices["hh_hv_ratio_db"], cmap="plasma", aspect="equal")
        axes[1, 1].set_title("HH/HV Ratio (dB)")

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(f"Polarimetric Analysis Summary — {STUDY_AREA}", fontsize=14)
plt.tight_layout()
plt.savefig("polarimetric_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Cleanup

Close the HDF5 file handle and summarize.

In [ ]:
h5.close()
logger.info("HDF5 file handle closed.")
logger.info("Notebook 05 — Polarimetric Analysis complete.")
logger.info("Outputs saved: polarimetric_summary.png")